# MSE802 Quantum Computing: Assessment 2

## Task 3: Quantum Tic-Tac-Toe

The objective of this task is to analyse the provided quantum tic-tac-toe game, to complete the instructions that were deliberately removed from the template, and subsequently to execute the game on the local Qiskit simulator (Qiskit Contributors, n.d.). The completed code is presented in this notebook, whereas the detailed analysis of each decision, together with the supporting screenshots, is presented in the accompanying Word document.

## The Board as a Nine-Qubit Circuit

Each cell of the 3x3 board corresponds to one qubit of a nine-qubit circuit, and every move appends a gate to that circuit; the state of the game is therefore a quantum state rather than a fixed grid of symbols. The O and X moves apply rotations that turn a cell toward $|0\rangle$ or $|1\rangle$ respectively, the Not move applies a Pauli-X gate, and the SWAP move exchanges two cells. Consequently, a cell may remain in a superposition of both players until the Measure move executes the circuit, at which point every cell collapses to a definite owner and the winning triples are counted (Nielsen & Chuang, 2010).

In [1]:
'''
Only needed on a fresh environment such as Google Colab. On a local kernel that
already provides these packages (for example the qiskEnv conda environment used
for the other tasks) leave them commented out, because the shell "pip" may not
belong to the same interpreter that is running the notebook.

%pip install qiskit qiskit-aer pylatexenc ipywidgets --quiet
'''

'\nOnly needed on a fresh environment such as Google Colab. On a local kernel that\nalready provides these packages (for example the qiskEnv conda environment used\nfor the other tasks) leave them commented out, because the shell "pip" may not\nbelong to the same interpreter that is running the notebook.\n\n%pip install qiskit qiskit-aer pylatexenc ipywidgets --quiet\n'

In [2]:
from qiskit import *
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector, plot_state_city
from qiskit_aer import AerSimulator

# The board is laid out with plain ipywidgets rather than google.colab.widgets,
# so the game runs unchanged on a local Jupyter kernel (VS Code) and on Colab.
import ipywidgets
from ipywidgets import Button, HBox, VBox, HTML, Image, Output, Layout
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import math
import random
import io


## 1. The Provided Game Code

The game comprises the two classes below: `Board` holds the nine-qubit game circuit together with the printable board labels, whereas `Game` builds the interactive front end with ipywidgets (Jupyter Widgets Contributors, n.d.) and connects each button to its corresponding move.

## 2. Modifications to the Provided Code

The template was distributed with several instructions deliberately missing; the following completions and corrections were made in order to obtain a working game, and each decision is analysed in detail in the accompanying Word document.

1. The four branches of `make_move` updated the board labels but never touched the circuit. The missing gates were added: an X gate for the Not move, a rotation `ry(-pi/4)` toward $|0\rangle$ for the O move, a rotation `ry(+pi/4)` toward $|1\rangle$ for the X move, and a `swap` between the selected pair of cells for the SWAP move.
2. The tuple in `countWinners` was empty; the eight winning triples of the 3x3 grid (three rows, three columns and two diagonals) were filled in.
3. The original `measure` assigned a measured 0 to player X and a measured 1 to player O, which contradicts the comments in `make_move`, since the O button is described there as a rotation toward $|0\rangle$. The mapping was flipped so that 0 is an O tile and 1 is an X tile; the comments were treated as authoritative because they describe the designer's intent. An empirical check over 300 rounds confirms the correction: pressing O twice on a cell produced an O tile in 0/300 runs under the original mapping and in 300/300 runs after the fix.
4. The original SWAP branch contained an unreachable test (`if self.target == cell` inside a branch already guarded by `self.target != cell`), and with `target` initialised to -1 an unpaired SWAP silently swapped with cell 8 through negative indexing; the dead test was replaced with a real `target < 0` guard.
5. The layout used `google.colab.widgets`, which exists only on Colab, and `clear_output()` does not clear reliably inside button callbacks on a local kernel. The interface was therefore rebuilt with plain `ipywidgets`, and the display is updated by reassigning the children of a single `VBox` container so that the board is replaced rather than appended on every click; consequently the game runs unchanged in VS Code and on Colab.

In [3]:

class Board:
    """Holds the 9-qubit game circuit and the printable board labels.

    Qubit i and classical bit i both correspond to board cell i (0 top-left,
    8 bottom-right). Convention used here: a cell measured as |0> belongs to
    O, a cell measured as |1> belongs to X.
    """

    def __init__(self):
        self.qc = QuantumCircuit(9, 9)
        self.function = ''
        self.target = -1
        self.tab = []
        self.winsX = 0
        self.winsO = 0

        # every cell starts in an equal superposition, so before any move each
        # tile is 50/50 between O and X
        for idx in range(0, 9):
            self.tab.append({'default':str(idx), 'player':' '})
            self.qc.reset(idx)
            self.qc.h(idx)
        self.qc.barrier()

    def make_move(self, cell):
        """Append the gate for the currently selected move onto cell `cell`."""
        if self.function == 'Not':
            # X gate: flips |0> <-> |1>, so the tile changes hands
            self.qc.x(cell)
            self.tab[int(cell)]['player'] += 'N - '
        elif self.function == 'O':
            # add a rotation toward |0>: Ry(-pi/4) biases the tile toward O.
            # From |+> two such moves land exactly on |0>, a certain O tile.
            self.qc.ry(-np.pi / 4, cell)
            self.tab[int(cell)]['player'] += "O - "
        elif self.function == 'X':
            # add a rotation toward |1>: Ry(+pi/4) biases the tile toward X.
            # From |+> two such moves land exactly on |1>, a certain X tile.
            self.qc.ry(np.pi / 4, cell)
            self.tab[int(cell)]['player'] += "X - "
        elif self.function == 'SWAP' and self.target != cell:
            if self.target < 0:
                # no source cell picked yet, so there is nothing to swap with
                self.target = -1
            else:
                # add a swap gate: exchanges the states of the two cells,
                # entanglement included
                self.qc.swap(self.target, cell)
                self.tab[int(cell)]['player'] += "S - "
                self.tab[int(self.target)]['player'] += "S - "
                self.target = -1

    def results(self):
        # hand back the circuit that produced the round, then start a clean one
        fig = self.qc.draw('mpl')
        self.qc = QuantumCircuit(9, 9)
        return fig

    def display(self):
        # hand back the circuit as it stands; Game decides where to draw it
        return self.qc.draw('mpl')

    def measure(self):
        self.qc.barrier()
        for i in range(0,9):
            self.qc.measure(i, i)

        simulator = AerSimulator()
        job = simulator.run(self.qc, shots=1)
        counts = job.result().get_counts()
        output = list(counts.keys())[0]

       # output = job.result().get_memory()[0]

        # Qiskit returns the bitstring little-endian, so classical bit i sits at
        # index 9-1-i. |0> is an O tile, |1> is an X tile, matching the
        # rotation directions used in make_move.
        for i in range(0,9):
            if output[9-1-i] == '0':
                self.tab[i]['player'] = 'O'
            else:
                self.tab[i]['player'] = 'X'
        self.winsX = self.countWinners('X')
        self.winsO = self.countWinners('O')

    def countWinners(self, player):
        # the 8 triples of win conditions: 3 rows, 3 columns, 2 diagonals
        winners = ((0, 1, 2), (3, 4, 5), (6, 7, 8),
                   (0, 3, 6), (1, 4, 7), (2, 5, 8),
                   (0, 4, 8), (2, 4, 6))
        wins = 0
        for i in range(len(winners)):
            won = True
            for j in range(len(winners[0])):
                if not self.tab[winners[i][j]]['player'] == player:
                    won = False
            if won:
                wins = wins + 1
        return wins

    def new(self):
        # discard the previous round entirely, then re-prepare the 9 tiles
        self.qc = QuantumCircuit(9, 9)
        self.function = ''
        self.target = -1
        self.tab.clear()
        for idx in range(0,9):
           self.tab.append({'default':str(idx), 'player':' '})
           self.qc.reset(idx)
           self.qc.h(idx)
        self.qc.barrier()


In [4]:
class Game:
    """The interactive front end.

    The whole interface is one VBox whose children are rebuilt on every click.
    Rendering through widget state rather than through print/display keeps the
    board from accumulating copies: a local kernel does not reliably honour
    clear_output() from inside a button callback, whereas reassigning .children
    always replaces what is on screen.
    """

    def __init__(self):
        self.selecting = False
        self.board = Board()

        # one button per board cell, labelled with its index 0..8
        self.boardbutton_list = []
        for i in range(0,9):
            button = Button(description=str(i))
            button.on_click(self.handle_game)
            self.boardbutton_list.append(button)

        # the move menu shown above the board
        self.funcbutton_list = []
        self.newButton('Measure')
        self.newButton('Not')
        self.newButton('O')
        self.newButton('X')
        self.newButton('SWAP')

        self.log = Output()          # anything that goes wrong in a callback
        self.app = VBox([])
        display(self.app)
        self.app.children = (self.printmenu(), self.printBoard(), self.log)

    def newButton(self, name):
        function = Button(description=name, layout=Layout(width='86px', height='30px'))
        function.on_click(self.handle_game)
        self.funcbutton_list.append(function)

    def handle_game(self, b):
        # Every button routes through here. int(b.description) succeeds only for
        # the 9 numbered cells, so a menu button raises ValueError and is handled
        # below, where it is recorded as the move to apply on the next cell click.
        with self.log:
            try:
                if b.description == 'Measure':
                    self.board.measure()
                    self.app.children = (self.replay(), self.scoreboard(),
                                         self.printBoard(),
                                         self.figure(self.board.results()), self.log)

                if b.description == 'Replay':
                    self.board.new()
                    self.app.children = (self.printmenu(), self.printBoard(), self.log)

                if int(b.description) >= 0:
                    if self.selecting:
                        # first cell of a SWAP: remember it, wait for the second
                        self.board.target = int(b.description)
                    else:
                        self.board.make_move(int(b.description))
                        self.app.children = (self.printmenu(), self.printBoard(),
                                             self.figure(self.board.display()), self.log)
                self.selecting = False
            except ValueError:
                # a menu button was pressed: record the chosen move
                self.board.function = b.description
                self.selecting = False
                if self.board.function == 'SWAP':
                    self.selecting = True

    def figure(self, fig):
        # a matplotlib figure cannot be a widget child, so render it to a PNG
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', dpi=96)
        plt.close(fig)
        return Image(value=buf.getvalue(), format='png',
                     layout=Layout(max_width='100%'))

    def printmenu(self):
        return HBox(self.funcbutton_list)

    def scoreboard(self):
        return HTML("<pre>X wins: %d    O wins: %d</pre>"
                    % (self.board.winsX, self.board.winsO))

    def replay(self):
        rep = Button(description="Replay")
        rep.on_click(self.handle_game)
        return rep

    def printBoard(self):
        # 3x3 grid; each cell shows its move history above its own button
        side = int(np.sqrt(9))
        rows = []
        for row in range(side):
            cells = []
            for col in range(side):
                i = col + row * side
                label = HTML("<pre style='margin:0;text-align:center'>%s</pre>"
                             % self.board.tab[i]['player'])
                cells.append(VBox([label, self.boardbutton_list[i]],
                                  layout=Layout(border='1px solid gray', padding='4px')))
            rows.append(HBox(cells))
        return VBox(rows)


## 3. Playing the Game on the Qiskit Simulator

In this iteration the game is executed on the local Qiskit simulator; each round proceeds by appending gates to the game circuit and ends when the Measure move runs the circuit, collapses every cell to a definite owner, and counts the winning triples.

### 3.1 How to Play

In order to make a move, one of the five options above the board is selected and the target location on the board is then clicked; the SWAP move requires two board locations. Each move appends a gate to the game circuit, as follows:

* **Measure** ends the round and executes the game circuit on the simulator; the win conditions are then counted and displayed.
* **Not** flips an owned tile to the other player; if the tile is not currently owned, this move has no effect.
* **O** rotates the selected cell toward a tile owned by O.
* **X** rotates the selected cell toward a tile owned by X.
* **SWAP** exchanges the locations of two tiles.

As the game progresses, both the game circuit and the board are displayed; the board records the sequence of moves but is not fully accurate, whereas the circuit is the true state of the game.

In [5]:
game = Game()

VBox()

## References

Jupyter Widgets Contributors. (n.d.). *ipywidgets: Interactive HTML widgets for Jupyter* (Version 8.1.8) [Computer software]. Project Jupyter. Retrieved July 26, 2026, from https://ipywidgets.readthedocs.io

Nielsen, M. A., & Chuang, I. L. (2010). *Quantum computation and quantum information* (10th anniversary ed.). Cambridge University Press.

Qiskit Contributors. (n.d.). *Qiskit: An open-source framework for quantum computing* (Version 2.4.0) [Computer software]. IBM Quantum. Retrieved July 26, 2026, from https://www.ibm.com/quantum/qiskit